# Petición HTTP a la API de TMBD

In [2]:
import os
import requests
import pandas as pd
import getpass


Empezamos solicitando la clave API para la página TMBD (guardado en el documento .env) y cargamos los datos de `movies.csv` y `links.csv`

In [ ]:

TMDB_API_KEY = os.getenv('TMDB_API_KEY')

In [8]:
movies = pd.read_csv('movies.csv')
links = pd.read_csv('links.csv')

Ahora construimos un `DataFrame` (`movies10`) con 10 peliculas del dataset original.

In [19]:
movies_links = pd.merge(movies, links[['movieId', 'tmdbId']], on='movieId', how='left')
movies10 = movies_links.dropna(subset=['tmdbId']).head(10).copy()

Creamos la función `fetch_movie_details(tmdb_id)` que haga una petición a la API y devuelva `overview` y `homepage` (en el caso de recibir valores nulos, se devolverá texto vacío)

In [20]:
def fetch_movie_details(tmdb_id):
    # Endpoint exacto del enunciado
    url = f"https://api.themoviedb.org/3/movie/{int(tmdb_id)}?api_key={TMDB_API_KEY}"
    
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            # Obtenemos los campos. Si vienen nulos o no existen, usamos un texto vacío ("")
            overview = data.get('overview') or ""
            homepage = data.get('homepage') or ""
            
            # Devolvemos una Serie de Pandas para poder crear dos columnas a la vez luego
            return pd.Series([overview, homepage])
        else:
            return pd.Series(["", ""])
    except Exception as e:
        print(f"Error con ID {tmdb_id}: {e}")
        return pd.Series(["", ""])

Por último, recorremos movies10 y añadimos a cada registro las columnas de `overview` y `homepage` en el `DataFrame`

In [25]:
movies10[['overview', 'homepage']] = movies10['tmdbId'].apply(fetch_movie_details)

print("Mostrando las columnas solicitadas:")
display(movies10[['title', 'tmdbId', 'overview', 'homepage']])

Mostrando las columnas solicitadas:


,title,tmdbId,overview,homepage
0,Toy Story (1995),862.0,"Led by Woody, Andy's toys live happily in his ...",http://toystory.disney.com/toy-story
1,Jumanji (1995),8844.0,When siblings Judy and Peter discover an encha...,http://www.sonypictures.com/movies/jumanji/
2,Grumpier Old Men (1995),15602.0,A family wedding reignites the ancient feud be...,
3,Waiting to Exhale (1995),31357.0,"Cheated on, mistreated and stepped on, the wom...",
4,Father of the Bride Part II (1995),11862.0,Just when George Banks has recovered from his ...,
5,Heat (1995),949.0,Obsessive master thief Neil McCauley leads a t...,https://www.20thcenturystudios.com/movies/heat
6,Sabrina (1995),11860.0,"After her return from school in Paris, a playb...",
7,Tom and Huck (1995),45325.0,"A mischievous young boy, Tom Sawyer, witnesses...",
8,Sudden Death (1995),9091.0,When a man's daughter is suddenly taken during...,
9,GoldenEye (1995),710.0,When a powerful secret defense system is stole...,https://mgm.com/movies/goldeneye
